In [9]:
import pandas as pd
from typing import Dict, List, Set, Tuple, Optional, Any


In [11]:
def load_data(file_path: str) -> Optional[pd.DataFrame]:
    """
    Load data from an Excel file and perform initial data cleaning.
    
    Args:
        file_path: Path to the Excel file containing material data
        
    Returns:
        DataFrame containing loaded data or None if loading fails
        
    Raises:
        FileNotFoundError: If the specified file doesn't exist
        ValueError: If there are issues with data types or formatting
    """
    try:
        df = pd.read_excel(file_path)
        
        df['year'] = df['year'].astype(int)
        df['produced_material'] = df['produced_material'].astype(str)
        df['component_material'] = df['component_material'].astype(str).replace('nan', pd.NA)
        df['plant_id'] = df['plant_id'].astype(str)
        
        print(f"Successfully loaded {len(df)} rows from {file_path}")
        print("Sample data:")
        print(df.head())
        
        return df
    
    except FileNotFoundError:
        print(f"Error: File {file_path} not found")
        return None
    except Exception as e:
        print(f"Error loading file: {str(e)}")
        return None


In [13]:
def aggregate_data(df: pd.DataFrame) -> Optional[pd.DataFrame]:
    """
    Aggregate material data by key dimensions and sum quantities.
    
    Args:
        df: Raw DataFrame containing material data
        
    Returns:
        Aggregated DataFrame or None if aggregation fails
        
    The function groups data by:
    - plant_id, year, produced_material, component_material
    - release types and production types
    And sums the quantities for produced and component materials.
    """
    try:
        df_agg = df.groupby([
            'plant_id', 'year', 'produced_material', 'component_material',
            'produced_material_release_type', 'produced_material_production_type',
            'component_material_release_type', 'component_material_production_type'
        ], dropna=False).agg({
            'produced_material_quantity': 'sum',
            'component_material_quantity': 'sum'
        }).reset_index()
        
        print(f"Aggregated {len(df_agg)} unique material combinations")
        return df_agg
    
    except Exception as e:
        print(f"Error in aggregation: {str(e)}")
        return None


In [15]:
def identify_fin_materials(df_agg: pd.DataFrame) -> Set[str]:
    """
    Identify all FIN (finished) materials in the dataset.
    
    Args:
        df_agg: Aggregated DataFrame containing material data
        
    Returns:
        Set of material IDs that are FIN materials
    """
    try:
        fin_materials = set(df_agg[df_agg['produced_material_release_type'] == 'FIN']['produced_material'])
        print(f"Found {len(fin_materials)} FIN materials")
        return fin_materials
    except Exception as e:
        print(f"Error identifying FIN materials: {str(e)}")
        return set()


In [17]:
def create_lookup_dictionaries(df_agg: pd.DataFrame) -> Dict[str, Dict]:
    """
    Create lookup dictionaries for efficient hierarchy traversal.
    
    Args:
        df_agg: Aggregated DataFrame containing material data
        
    Returns:
        Dictionary containing several lookup dictionaries:
        - release_type: Material ID → release type
        - prod_type: Material ID → production type
        - comp_release_type: Component ID → release type
        - comp_prod_type: Component ID → production type
        - components: Material ID → {component ID: quantity}
        - prod_quantity: (plant, year, material) → production quantity
    """
    try:
        prod_quantities = df_agg.groupby(['plant_id', 'year', 'produced_material'])['produced_material_quantity'].sum().to_dict()
        
        component_df = df_agg[df_agg['component_material'].notna()]
        lookup_dicts = {
            'release_type': df_agg[['produced_material', 'produced_material_release_type']]
                .drop_duplicates('produced_material')
                .set_index('produced_material')['produced_material_release_type'].to_dict(),
            'prod_type': df_agg[['produced_material', 'produced_material_production_type']]
                .drop_duplicates('produced_material')
                .set_index('produced_material')['produced_material_production_type'].to_dict(),
            'comp_release_type': component_df[['component_material', 'component_material_release_type']]
                .drop_duplicates('component_material')
                .set_index('component_material')['component_material_release_type'].to_dict(),
            'comp_prod_type': component_df[['component_material', 'component_material_production_type']]
                .drop_duplicates('component_material')
                .set_index('component_material')['component_material_production_type'].to_dict(),
            'components': component_df.groupby('produced_material', group_keys=False).apply(
                lambda x: dict(zip(x['component_material'], x['component_material_quantity'])),
                include_groups=False
            ).to_dict(),
            'prod_quantity': prod_quantities
        }
        
        print("Successfully created lookup dictionaries")
        return lookup_dicts
    
    except Exception as e:
        print(f"Error creating lookup dictionaries: {str(e)}")
        return {}


In [19]:
def build_hierarchy(
    fin_material: str,
    plant: str,
    year: int,
    lookup_dicts: Dict[str, Dict],
    result_list: List[Dict],
    current_material: Optional[str] = None,
    path: Optional[List[str]] = None
) -> None:
    """
    Recursively build the material hierarchy starting from a FIN material.
    
    Args:
        fin_material: The FIN material ID to start from
        plant: Plant ID
        year: Year
        lookup_dicts: Dictionary containing all lookup dictionaries
        result_list: List to accumulate results
        current_material: Current material being processed (None for root)
        path: List of material IDs in the current path (for cycle detection)
        
    This function recursively traverses the material hierarchy and adds each
    valid component to the result list, maintaining the proper order.
    """
    if path is None:
        path = []
    
    if current_material is None:
        components = lookup_dicts['components'].get(fin_material, {})
        for component in sorted(components.keys()): 
            build_hierarchy(fin_material, plant, year, lookup_dicts, result_list, component, [fin_material])
    else:
        path = path + [current_material]
        
        if len(path) != len(set(path)):
            print(f"Warning: Cycle detected in material hierarchy: {' -> '.join(path)}")
            return
        
        fin_release_type = lookup_dicts['release_type'].get(fin_material, 'Unknown')
        current_release_type = lookup_dicts['release_type'].get(current_material, 'Unknown')
        
        if fin_release_type != current_release_type:
            components = lookup_dicts['components'].get(current_material, {})
            
            for component in sorted(components.keys()):
                comp_quantity = components[component]
                comp_release_type = lookup_dicts['comp_release_type'].get(component, 'Unknown')
                
                row = {
                    'plant': plant,
                    'year': year,
                    'fin_material_id': fin_material,
                    'fin_material_release_type': fin_release_type,
                    'fin_material_production_type': lookup_dicts['prod_type'].get(fin_material, None),
                    'fin_production_quantity': lookup_dicts['prod_quantity'].get((plant, year, fin_material), 0),
                    'prod_material_id': current_material,
                    'prod_material_release_type': current_release_type,
                    'prod_material_production_type': lookup_dicts['prod_type'].get(current_material, None),
                    'prod_material_production_quantity': lookup_dicts['prod_quantity'].get((plant, year, current_material), 0),
                    'component_id': component,
                    'component_material_release_type': comp_release_type,
                    'component_material_production_type': lookup_dicts['comp_prod_type'].get(component, None),
                    'component_consumption_quantity': comp_quantity
                }
                
                result_list.append(row)
                build_hierarchy(fin_material, plant, year, lookup_dicts, result_list, component, path)


In [33]:
def process_fin_materials(
    df_agg: pd.DataFrame,
    fin_materials: Set[str],
    lookup_dicts: Dict[str, Dict]
) -> pd.DataFrame:
    """
    Process all FIN materials and build their hierarchies.
    
    Args:
        df_agg: Aggregated DataFrame
        fin_materials: Set of FIN material IDs
        lookup_dicts: Dictionary containing lookup dictionaries
        
    Returns:
        DataFrame containing the exploded hierarchy for all FIN materials
    """
    fin_material_combinations = df_agg[df_agg['produced_material'].isin(fin_materials)][
        ['plant_id', 'year', 'produced_material']
    ].drop_duplicates()
    
    result_list = []
    for _, row in fin_material_combinations.iterrows():
        plant = row['plant_id']
        year = row['year']
        fin_material = row['produced_material']
        build_hierarchy(fin_material, plant, year, lookup_dicts, result_list)
    
    result_df = pd.DataFrame(result_list)
    
    result_df = result_df.sort_values([
        'plant', 'year', 'fin_material_id', 'prod_material_id', 'component_id'
    ]).reset_index(drop=True)
    
    return result_df


In [39]:
"""Main Material explosion analysis"""
file_path = 'C:/Users/egor2/trainee_de/task2_data/task_2_data_ex.xlsx'
output_path = 'bom_explosion_output.xlsx'
    
df = load_data(file_path)
if df is None:
    raise SystemExit("Failed to load data")
    
df_agg = aggregate_data(df)
if df_agg is None:
    raise SystemExit("Failed to aggregate data")
    
fin_materials = identify_fin_materials(df_agg)
if not fin_materials:
    print("No FIN materials found - nothing to process")
else:
    lookup_dicts = create_lookup_dictionaries(df_agg)
    if not lookup_dicts:
        print("Failed to create lookup dictionaries - cannot proceed")
    else:
        result_df = process_fin_materials(df_agg, fin_materials, lookup_dicts)
        print(f"Total rows: {len(result_df)}")
        print("Sample output:")
        print(result_df.head())
        
        result_df.to_excel(output_path, index=False)
        print(f"\nResults saved to {output_path}")

Successfully loaded 1466 rows from C:/Users/egor2/trainee_de/task2_data/task_2_data_ex.xlsx
Sample data:
   year  month produced_material  produced_material_production_type  \
0  2024      1             10000                               8002   
1  2024      1             50000                               8002   
2  2024      1             50000                               8002   
3  2024      1             50000                               8002   
4  2024      1             80070                               8007   

  produced_material_release_type  produced_material_quantity  \
0                            FIN                         990   
1                           PROD                         859   
2                           PROD                         859   
3                           PROD                         859   
4                           PROD                         929   

  component_material  component_material_production_type  \
0              50000   